# Week 4 — Cancellation and Customer Rating Analysis

This notebook reads the checked-in Olist source files and writes candidate CSVs to `data/exports`. It keeps cancellation status, review-record distributions, and order-level delay/rating comparisons at their appropriate grains. Delays use calendar dates; valid ratings are integer scores 1–5 linked by `order_id`. Each rated order contributes once to a delivery-group mean after averaging its linked valid review scores. The descriptive comparison is not causal.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
EXPORTS = ROOT / "data" / "exports"
EXPORTS.mkdir(parents=True, exist_ok=True)

orders = pd.read_csv(
    RAW / "olist_orders_dataset.csv",
    parse_dates=["order_delivered_customer_date", "order_estimated_delivery_date"],
)
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")

assert orders["order_id"].notna().all()
assert orders["order_id"].is_unique
assert customers["customer_id"].notna().all()
assert customers["customer_id"].is_unique
assert reviews["order_id"].notna().all()
assert reviews["review_id"].notna().all()
assert reviews["order_id"].isin(orders["order_id"]).all()
assert not reviews.duplicated(["order_id", "review_id"]).any()
valid_reviews = reviews.loc[reviews["review_score"].between(1, 5)].copy()

print(f"Orders: {len(orders):,} unique; reviews: {len(reviews):,}; customers: {len(customers):,}")

## 1. Cancellation and order-status distribution

The cancellation numerator is exactly `order_status == 'canceled'`; all valid unique orders are the denominator. `unavailable` remains a separate status. State rates are calculated within state and include their order counts.

In [ ]:
orders_with_state = orders.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left",
    validate="many_to_one",
)
orders_with_state["customer_state"] = (
    orders_with_state["customer_state"].fillna("").str.strip().str.upper().replace("", "Unknown")
)
total_orders = len(orders)
cancellation_rows = [{
    "record_type": "overall", "dimension": "all_orders", "customer_state": "",
    "order_status": "", "order_count": total_orders,
    "canceled_orders": int(orders["order_status"].eq("canceled").sum()),
    "cancellation_rate_pct": round(100 * orders["order_status"].eq("canceled").mean(), 2),
    "pct_of_all_orders": 100.0, "pct_of_state_orders": "",
}]

for status, count in orders["order_status"].value_counts().items():
    cancellation_rows.append({
        "record_type": "status", "dimension": status, "customer_state": "",
        "order_status": status, "order_count": int(count),
        "canceled_orders": int((orders["order_status"].eq(status) & orders["order_status"].eq("canceled")).sum()),
        "cancellation_rate_pct": "",
        "pct_of_all_orders": round(100 * count / total_orders, 2),
        "pct_of_state_orders": "",
    })

for state, frame in orders_with_state.groupby("customer_state", sort=True):
    state_n = len(frame)
    state_canceled = int(frame["order_status"].eq("canceled").sum())
    cancellation_rows.append({
        "record_type": "state", "dimension": state, "customer_state": state,
        "order_status": "", "order_count": state_n, "canceled_orders": state_canceled,
        "cancellation_rate_pct": round(100 * state_canceled / state_n, 2),
        "pct_of_all_orders": round(100 * state_n / total_orders, 2),
        "pct_of_state_orders": "",
    })

for state, frame in orders_with_state.groupby("customer_state", sort=True):
    state_n = len(frame)
    for status, status_n in frame["order_status"].value_counts().sort_index().items():
        cancellation_rows.append({
            "record_type": "state_status", "dimension": f"{state}:{status}",
            "customer_state": state, "order_status": status,
            "order_count": int(status_n),
            "canceled_orders": int((frame["order_status"].eq(status) & frame["order_status"].eq("canceled")).sum()),
            "cancellation_rate_pct": "",
            "pct_of_all_orders": round(100 * status_n / total_orders, 2),
            "pct_of_state_orders": round(100 * status_n / state_n, 2),
        })

cancellation = pd.DataFrame(cancellation_rows)
assert cancellation.loc[cancellation.record_type.eq("status"), "order_count"].sum() == total_orders
assert int(cancellation.loc[cancellation.record_type.eq("overall"), "canceled_orders"].iloc[0]) == 625
cancellation.to_csv(EXPORTS / "week4_cancellation_analysis.csv", index=False)
display(cancellation[cancellation.record_type.isin(["overall", "status"])])

## 2. Review-score distribution and coverage

The score distribution counts review records. Coverage and the overall mean are computed at order grain: valid scores are averaged within `order_id` first, then each order contributes once.

In [ ]:
review_orders = valid_reviews.groupby("order_id", as_index=False).agg(
    valid_review_count=("review_score", "size"),
    avg_review_score=("review_score", "mean"),
)
assert reviews["order_id"].isin(orders["order_id"]).all()

rating_rows = []
valid_review_total = len(reviews)
for score in range(1, 6):
    score_rows = reviews[reviews.review_score.eq(score)]
    rating_rows.append({
        "record_type": "score_distribution", "category": str(score),
        "review_records": len(score_rows), "distinct_orders": score_rows.order_id.nunique(),
        "pct_review_records": round(100 * len(score_rows) / valid_review_total, 2),
        "order_status": "", "total_orders": "", "orders_with_review": "",
        "orders_without_review": "", "review_coverage_pct": "",
        "delivery_group": "", "delivered_orders_with_valid_dates": "",
        "rated_orders": "", "orders_without_valid_rating": "",
        "mean_order_level_review_score": "",
    })

for category, mask in [
    ("missing_score", reviews.review_score.isna()),
    ("invalid_score", reviews.review_score.notna() & ~reviews.review_score.between(1, 5)),
]:
    score_rows = reviews.loc[mask]
    rating_rows.append({
        "record_type": "score_distribution", "category": category,
        "review_records": len(score_rows), "distinct_orders": score_rows.order_id.nunique(),
        "pct_review_records": round(100 * len(score_rows) / valid_review_total, 2),
        "order_status": "", "total_orders": "", "orders_with_review": "",
        "orders_without_review": "", "review_coverage_pct": "",
        "delivery_group": "", "delivered_orders_with_valid_dates": "",
        "rated_orders": "", "orders_without_valid_rating": "",
        "mean_order_level_review_score": "",
    })

valid_rated_orders = review_orders.order_id.nunique()
rating_rows.extend([
    {"record_type": "overall_rating_summary", "category": "all_orders",
     "review_records": valid_review_total, "distinct_orders": valid_rated_orders,
     "pct_review_records": "", "order_status": "", "total_orders": total_orders,
     "orders_with_review": valid_rated_orders,
     "orders_without_review": total_orders - valid_rated_orders,
     "review_coverage_pct": round(100 * valid_rated_orders / total_orders, 2),
     "delivery_group": "", "delivered_orders_with_valid_dates": "",
     "rated_orders": "", "orders_without_valid_rating": total_orders - valid_rated_orders,
     "mean_order_level_review_score": round(review_orders.avg_review_score.mean(), 4)},
    {"record_type": "overall_review_coverage", "category": "all_orders",
     "review_records": valid_review_total, "distinct_orders": valid_rated_orders,
     "pct_review_records": "", "order_status": "", "total_orders": total_orders,
     "orders_with_review": valid_rated_orders,
     "orders_without_review": total_orders - valid_rated_orders,
     "review_coverage_pct": round(100 * valid_rated_orders / total_orders, 2),
     "delivery_group": "", "delivered_orders_with_valid_dates": "",
     "rated_orders": "", "orders_without_valid_rating": "",
     "mean_order_level_review_score": ""},
])

review_flag = orders[["order_id", "order_status"]].merge(
    reviews[["order_id"]].drop_duplicates().assign(has_review=1),
    on="order_id", how="left", validate="one_to_one",
)
for status, frame in review_flag.groupby("order_status", sort=True):
    count = len(frame)
    covered = int(frame.has_review.fillna(0).sum())
    rating_rows.append({
        "record_type": "review_coverage_by_status", "category": "",
        "review_records": "", "distinct_orders": "", "pct_review_records": "",
        "order_status": status, "total_orders": count, "orders_with_review": covered,
        "orders_without_review": count - covered,
        "review_coverage_pct": round(100 * covered / count, 2),
        "delivery_group": "", "delivered_orders_with_valid_dates": "",
        "rated_orders": "", "orders_without_valid_rating": "",
        "mean_order_level_review_score": "",
    })

rating = pd.DataFrame(rating_rows)
display(rating[rating.record_type.eq("score_distribution")])

## 3. Delayed versus on-time ratings

Only delivered orders with both actual and estimated dates are eligible. Review rows must link to the order and have a score from 1 to 5. Group means use per-order averages. The score distribution retains valid linked review-record grain and reports distinct order counts alongside records.

In [ ]:
eligible = orders.loc[
    orders.order_status.eq("delivered")
    & orders.order_delivered_customer_date.notna()
    & orders.order_estimated_delivery_date.notna(),
    ["order_id", "order_delivered_customer_date", "order_estimated_delivery_date"],
].copy()
eligible["delivery_group"] = np.where(
    eligible.order_delivered_customer_date.dt.normalize()
    > eligible.order_estimated_delivery_date.dt.normalize(),
    "delayed", "on_time",
)
assert eligible.order_id.is_unique

linked_reviews = reviews.merge(
    eligible[["order_id", "delivery_group"]],
    on="order_id", how="inner", validate="many_to_one",
)
valid_linked_reviews = linked_reviews.loc[
    linked_reviews.review_score.between(1, 5)
].copy()
per_order = valid_linked_reviews.groupby(
    ["order_id", "delivery_group"], as_index=False
).agg(
    avg_order_score=("review_score", "mean"),
    valid_review_records=("review_score", "size"),
)
group_sizes = eligible.groupby("delivery_group").size().rename("eligible_delivered_orders")
group_ratings = per_order.groupby("delivery_group").agg(
    rated_orders=("order_id", "nunique"),
    mean_order_level_review_score=("avg_order_score", "mean"),
).join(group_sizes).reset_index()
group_ratings["orders_without_valid_rating"] = (
    group_ratings.eligible_delivered_orders - group_ratings.rated_orders
)
group_ratings["rated_order_coverage_pct"] = (
    100 * group_ratings.rated_orders / group_ratings.eligible_delivered_orders
).round(2)

comparison_rows = []
for row in group_ratings.itertuples(index=False):
    review_count = int(valid_linked_reviews.delivery_group.eq(row.delivery_group).sum())
    comparison_rows.append({
        "record_type": "group_summary", "delivery_group": row.delivery_group,
        "eligible_delivered_orders": int(row.eligible_delivered_orders),
        "rated_orders": int(row.rated_orders),
        "orders_without_valid_rating": int(row.orders_without_valid_rating),
        "rated_order_coverage_pct": row.rated_order_coverage_pct,
        "valid_review_records": review_count,
        "mean_order_level_review_score": round(row.mean_order_level_review_score, 4),
        "review_score": "", "distinct_orders_with_score": "",
        "pct_valid_review_records": "",
    })

for (delivery_group, score), frame in valid_linked_reviews.groupby(
    ["delivery_group", "review_score"], sort=True
):
    group_total = int(valid_linked_reviews.delivery_group.eq(delivery_group).sum())
    comparison_rows.append({
        "record_type": "score_distribution", "delivery_group": delivery_group,
        "eligible_delivered_orders": int(group_sizes.loc[delivery_group]),
        "rated_orders": int(group_ratings.loc[group_ratings.delivery_group.eq(delivery_group), "rated_orders"].iloc[0]),
        "orders_without_valid_rating": int(group_ratings.loc[group_ratings.delivery_group.eq(delivery_group), "orders_without_valid_rating"].iloc[0]),
        "rated_order_coverage_pct": float(group_ratings.loc[group_ratings.delivery_group.eq(delivery_group), "rated_order_coverage_pct"].iloc[0]),
        "valid_review_records": len(frame),
        "mean_order_level_review_score": "",
        "review_score": int(score),
        "distinct_orders_with_score": int(frame.order_id.nunique()),
        "pct_valid_review_records": round(100 * len(frame) / group_total, 2),
    })

comparison = pd.DataFrame(comparison_rows)
assert group_sizes.to_dict() == {"delayed": 6534, "on_time": 89936}
assert group_ratings.set_index("delivery_group").rated_orders.to_dict() == {"delayed": 6381, "on_time": 89443}
assert int(comparison.loc[comparison.record_type.eq("group_summary"), "eligible_delivered_orders"].sum()) == len(eligible)
comparison.to_csv(EXPORTS / "week4_delay_rating_comparison.csv", index=False)
for row in group_ratings.itertuples(index=False):
    rating_rows.append({
        "record_type": "delivery_group_rating", "category": "",
        "review_records": "", "distinct_orders": "", "pct_review_records": "",
        "order_status": "", "total_orders": "", "orders_with_review": "",
        "orders_without_review": "", "review_coverage_pct": "",
        "delivery_group": row.delivery_group,
        "delivered_orders_with_valid_dates": int(row.eligible_delivered_orders),
        "rated_orders": int(row.rated_orders),
        "orders_without_valid_rating": int(row.orders_without_valid_rating),
        "mean_order_level_review_score": round(row.mean_order_level_review_score, 4),
    })
rating = pd.DataFrame(rating_rows)
rating.to_csv(EXPORTS / "week4_rating_analysis.csv", index=False)
display(group_ratings)
print(f"Mean score gap (on time - delayed): {group_ratings.set_index('delivery_group').loc['on_time', 'mean_order_level_review_score'] - group_ratings.set_index('delivery_group').loc['delayed', 'mean_order_level_review_score']:.2f} points")

## 4. Operational actions and follow-up KPIs

- Pilot proactive recovery for a prespecified at-risk delivery cohort. Track on-time contact / qualifying at-risk orders (proposed service threshold: at least 95%), mean per-order valid score (baseline 2.2718 / 5, n=6,381 delayed rated orders), and late rate (6,534 / 96,470 = 6.77%). Compare mature cohorts and retain a suitable comparison group where feasible.
- Add a standardized cancellation reason (or explicit `unknown`) to every new cancellation; target 100% reason-field completion. Investigate São Paulo first by volume (327 / 41,746; 0.78%), without assuming geography is causal.
- Monitor valid-rating coverage: baseline 97.66% delayed (6,381 / 6,534) and 99.45% on time (89,443 / 89,936). Report both group sizes and do not impute missing ratings.
- Treat Roraima's 2.17% cancellation rate as a small-sample signal only (1 / 46), not a stable ranking.

See `reports/week4_operational_improvement_report.md` for the finalized candidate recommendations, KPI definitions, and release checks. These metrics are from checked-in raw files; reconcile to the approved database snapshot before publication.